# Step 2 — Preprocessing and time-aware split

| | |
|---|---|
| **Goal** | Turn the raw flows into clean, leakage-free **train / validation / test** sets, applying decisions that are each justified by the EDA. |
| **Inputs** | `data/interim/<day>.parquet` (step 0) |
| **Outputs** | `data/processed/{train,val,test}.parquet`, tables `results/tables/02_*.csv` |
| **Code** | `src/preprocess.py` (3 streamed passes) · settings in `src/config.py` |

### The decisions and their justification
Every row below answers a concrete finding from step 1. These are the arguments to give in the defense.

| # | Problem (EDA evidence) | Decision | Justification and alternatives rejected |
|---|---|---|---|
| **D1** | 12-hour clock: afternoon attacks appear 12 h early; no traffic at 06–07 h on any day. 14 flows dated 1970. | **Hours 01–07 → +12 h.** Drop the 14 invalid timestamps. | Uses only the capture schedule (08:00–17:59), **never the labels**. A label-based fix ("move the SSH flows to the afternoon") would leak the answer into a feature. |
| **D2** | 26.1% exact duplicates (FTP 99.97%, SlowHTTPTest 99.96%). | **Remove exact duplicates before splitting, keeping the first occurrence in time.** | Otherwise identical flows land in train *and* test, so the test measures memorisation. Rejected: deduplicating each split separately, which still leaves copies across splits. Cost: FTP and SlowHTTPTest shrink to ~55 flows each. This is reported, not hidden. |
| **D3** | 22,348 feature vectors labelled both Benign and Infiltration (+ a few attack/attack conflicts). | **Remove all flows whose feature vector carries more than one label.** Keep the Infiltration class, flagged as *unreliable*. | No classifier can get both versions right. Keeping them only adds irreducible error. Rejected: relabelling them "Benign", which would be an unverifiable guess. |
| **D4** | 95,760 flows with NaN/inf (zero duration; ~0% of attacks). ~15 flows with negative or impossible times. 8 constant features. | **Drop those rows; drop the constant features. Keep `-1` in `Init * Win Byts`.** | 0.59% of the data, almost only Benign, so dropping is nearly neutral. Imputing a rate for a zero duration would invent a value. `-1` is a meaningful sentinel ("not observed"). |
| — | 20/02: "LOIC-HTTP" flows during the documented LOIC-UDP window. | **Relabel "LOIC-HTTP" flows that use UDP (protocol 17) as LOIC-UDP.** | HTTP runs over TCP, so an HTTP flood over UDP is impossible. The rule uses a protocol fact, not the schedule. Both labels are in the DDoS family anyway, so the main experiments are unaffected. |
| **D6** | Each attack exists on 1–2 days, often in a single time window. | **Chronological 70/15/15 split inside every (day, class) group.** | Respects time (train on the start of each attack, test on its end) and keeps every class in every split. Rejected: random split (temporal leakage) and split by day (SSH, Hulk, etc. would exist only in test). |
| **D7** | ~10.6 M Benign flows after deduplication: too many for Colab CPU training. | **Downsample Benign to 1,000,000 flows in the training split only.** Validation and test keep the real proportions. | Undersampling the majority class for *training* is standard. Evaluating on the natural distribution keeps precision and the false-alarm rate honest. |
| **Labels** | SQLi 84, XSS 228 unique flows: per-class scores would be very unstable. | **Main target: 7 families** (Benign, DoS, DDoS, BruteForce, Web, Bot, Infiltration). The 15 fine labels are kept for per-attack analysis. | Stable estimates, a SOC-relevant granularity, and a realistic GAN target (Web ≈ 870 flows). Fine-grained recall is still reported. |
| **D5** | `Dst Port` almost identifies the class (testbed artefact). | **Kept in the data**, but excluded from the main models (decided in step 3), with one comparison run that includes it. | Lets us measure how much the models rely on this shortcut, which is also a robustness/evasion analysis. |

### Order matters
The operations run in this order, and each one depends on the previous ones:
**clean → fix the clock → sort chronologically → deduplicate (keeping the *first* flow in real time) → split → downsample.**
For example, deduplicating *before* fixing the clock would keep the wrong "first" copy, and splitting before
deduplicating would leak copies between splits.

## 1. Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/dija040/ids-project.git"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/ids-project")

    def git(*args):
        # Run a git command; if it fails, show git's own error message.
        r = subprocess.run(["git", *args], capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError("git " + " ".join(args) + " failed: " + r.stderr)
        return r.stdout.strip()

    if not (PROJECT_DIR / ".git").exists():
        git("clone", REPO_URL, str(PROJECT_DIR))
    else:
        # The Drive copy is only used to RUN the code (we edit it on the PC and push to
        # GitHub), so we force it to match GitHub exactly instead of merging with `pull`.
        # Untracked files (data/, generated results) are not touched.
        git("-C", str(PROJECT_DIR), "fetch", "origin")
        git("-C", str(PROJECT_DIR), "reset", "--hard", "origin/main")
    print("Code version:", git("-C", str(PROJECT_DIR), "log", "-1", "--format=%h %s"))
    os.environ["IDS_TMP_DIR"] = "/content/tmp"   # big intermediate files: temporary disk, not Drive
else:
    PROJECT_DIR = Path.cwd().resolve().parent
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import time
import numpy as np, pandas as pd
from src import config, preprocess, eda
config.set_seed(); config.ensure_dirs()
pd.set_option("display.max_rows", 100, "display.width", 200)
TAB = config.TABLES_DIR

## 2. Pass 1: clean each day

One day at a time. Each step is logged per class, so we can see **what each decision costs**.
The cleaned days go to a temporary folder (`IDS_TMP_DIR`); they are rebuilt whenever needed.

⏱️ About 10–15 minutes (most of it on the 20/02 file).

In [ ]:
clean_paths, logs = {}, []
for day in config.DAY_FILES:
    t0 = time.time()
    clean_paths[day] = config.TMP_DIR / "clean" / f"{day}.parquet"
    logs += preprocess.clean_day(day, config.INTERIM_DIR / f"{day}.parquet", clean_paths[day])
    print(f"{day} cleaned in {time.time()-t0:5.0f}s")

log = pd.DataFrame(logs)
log.to_csv(TAB / "02_cleaning_log.csv", index=False)
per_rule = (log[log["label"] != "ALL"].pivot_table(index="label", columns="rule", values="flows",
                                                  aggfunc="sum", fill_value=0))
print(f"Flows kept after cleaning: {log.loc[log['label'] == 'ALL', 'flows'].sum():,}")
per_rule

**Check:** the numbers should match the EDA: ~95.8k NaN/inf rows (mostly Benign and Infiltration), ~15
impossible times, 14 invalid timestamps. The relabelling line tells us whether the LOIC-UDP hypothesis was
right. If no line appears, no "LOIC-HTTP" flow used UDP, and the hypothesis is rejected: we would then report
that the documented LOIC-UDP attack of 20/02 is simply missing.

## 3. Pass 2: deduplicate, remove conflicts, split, downsample

Only three small columns of every flow are loaded (day, 64-bit fingerprint, label), about 200 MB for 16 M flows.

In [ ]:
t0 = time.time()
codes, stats = preprocess.plan_dedup_and_split(clean_paths)
print(f"planned in {time.time()-t0:.0f}s")
stats.to_csv(TAB / "02_flows_per_stage.csv")
stats.sort_values("after cleaning", ascending=False)

In [ ]:
# The same accounting, by attack family (the main target).
fam = stats.groupby(stats.index.map(config.LABEL_FAMILY)).sum().reindex(config.FAMILIES).fillna(0).astype(int)
denom = (fam["after cleaning"] - fam["removed: Benign train downsampling"]).replace(0, np.nan)
fam["kept %"] = (100 * fam[["train", "val", "test"]].sum(axis=1) / denom).round(1)
fam.to_csv(TAB / "02_flows_per_stage_family.csv")
fam

**How to read this.** `kept %` is the share of each family that survives deduplication and conflict removal
(Benign downsampling excluded). A low value means the class was mostly *copies* of a few flows. For FTP and
SlowHTTPTest it will be close to 0.03%, which is the most striking illustration of why scores on the raw dataset
are inflated.

## 4. Pass 3: write the train / validation / test files

In [ ]:
t0 = time.time()
rows = preprocess.write_splits(clean_paths, codes, config.PROCESSED_DIR)
for s, n in rows.items():
    f = config.PROCESSED_DIR / f"{s}.parquet"
    print(f"{s:5s}: {n:>10,} flows | {f.stat().st_size/1e6:,.0f} MB")
print(f"done in {time.time()-t0:.0f}s")

## 5. Verification: prove the split is leakage-free

A methodology is only as good as its checks. Three tests, all expected to pass:
1. **No identical flow in two splits**: no shared 64-bit fingerprint between train, validation and test.
2. **Chronology**: inside every (day, class) group, train flows come before validation flows, which come before test flows.
3. **Clock fix**: with the corrected clock, the observed attack windows should now match the official schedule.

In [ ]:
H = {s: pd.read_parquet(config.PROCESSED_DIR / f"{s}.parquet", columns=["fhash"])["fhash"]
     for s in config.SPLITS}
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    shared = np.intersect1d(H[a].to_numpy(), H[b].to_numpy()).size
    print(f"{a:5s} ∩ {b:4s}: {shared} shared feature vectors", "✅" if shared == 0 else "❌")
del H

In [ ]:
cols = ["day", config.TIMESTAMP_COLUMN, config.LABEL_COLUMN]
T = {s: pd.read_parquet(config.PROCESSED_DIR / f"{s}.parquet", columns=cols) for s in config.SPLITS}
g = {s: T[s].groupby(["day", config.LABEL_COLUMN], observed=True)[config.TIMESTAMP_COLUMN] for s in config.SPLITS}
order = pd.DataFrame({"train_last": g["train"].max(), "val_first": g["val"].min(),
                      "val_last": g["val"].max(), "test_first": g["test"].min()})
ok = ((order["train_last"] <= order["val_first"]) & (order["val_last"] <= order["test_first"])).fillna(True)
print(f"Chronological order respected in {ok.sum()} / {len(ok)} (day, class) groups", "✅" if ok.all() else "❌")
order[~ok]

In [ ]:
# Observed attack windows with the corrected clock vs the official schedule.
allT = pd.concat(T.values(), ignore_index=True)
win = (allT.groupby(["day", config.LABEL_COLUMN], observed=True)[config.TIMESTAMP_COLUMN]
           .agg(["min", "max", "size"]).reset_index())
doc = pd.DataFrame(config.DOCUMENTED_ATTACK_WINDOWS, columns=["raw_label", "day", "doc_start", "doc_end"])
doc[config.LABEL_COLUMN] = doc["raw_label"].map(config.LABEL_CANONICAL)
doc = doc.groupby(["day", config.LABEL_COLUMN]).agg(doc_start=("doc_start", "min"), doc_end=("doc_end", "max")).reset_index()
cmp = doc.merge(win, on=["day", config.LABEL_COLUMN], how="outer")
cmp["obs_first"] = cmp["min"].dt.strftime("%H:%M"); cmp["obs_last"] = cmp["max"].dt.strftime("%H:%M")
cmp = cmp[cmp[config.LABEL_COLUMN] != "Benign"][["day", config.LABEL_COLUMN, "doc_start", "doc_end", "obs_first", "obs_last", "size"]]
cmp.to_csv(TAB / "02_attack_windows_after_clock_fix.csv", index=False)
del allT
display(cmp.sort_values(["day", "doc_start"]))

## 6. The final datasets

In [ ]:
counts = pd.DataFrame({s: T[s][config.LABEL_COLUMN].value_counts() for s in config.SPLITS}).fillna(0).astype(int)
counts = counts.reindex(config.LABELS)
counts["family"] = counts.index.map(config.LABEL_FAMILY)
counts.to_csv(TAB / "02_split_counts_label.csv")

fam_counts = counts.groupby("family")[list(config.SPLITS)].sum().reindex(config.FAMILIES)
fam_counts["train_ratio_vs_benign"] = (fam_counts.loc["Benign", "train"] / fam_counts["train"]).round(1)
fam_counts.to_csv(TAB / "02_split_counts_family.csv")
display(fam_counts)
counts

## 7. What to look at, and what comes next

1. **Cleaning log:** was the LOIC-UDP relabelling triggered? How many flows did each rule remove?
2. **`kept %` per family:** which families were mostly duplicates? What does this say about published
   99%-accuracy results on CIC-IDS2018?
3. **Verification:** all three checks should show ✅. Paste them in the report: they *prove* the absence of
   leakage, which is the first item of the course's self-assessment checklist.
4. **Final counts:** how many Web flows are in train? This is what the GAN will learn from. Compare it with
   the other families: the imbalance ratio in train is now much lower than in the raw data, but still large for Web.

**Not done here, on purpose:** scaling and feature selection are *fitted on the training set* in the next steps,
so that no statistic of the validation/test data leaks into the models.

**Next step (03, feature selection):** remove redundant features (ρ ≈ 1 pairs from the EDA), rank features by
importance using the training split only, and decide on the `Dst Port` question (D5).

> 📝 Send me the outputs of sections 2, 3, 5 and 6, and fill in `reports/step_reports/02_preprocessing.md`.